# Figure2c rbp enrichment


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from typing import Set, Tuple, Dict
import warnings
warnings.filterwarnings('ignore')

GENE_RESULTS_CSV = "results/permutation_full/sle_vs_hc/slehc_gene_level_clean_summary.csv"
RBP_CELL_LINE_XLSX = "data/mmc2.xlsx"
FIG_DIR = "results/fig"

def load_gene_results(csv_path):
    df = pd.read_csv(csv_path)
    df['gene'] = df['gene'].astype(str).str.upper().str.strip()
    return df

def load_rbp_genes(xlsx_path):
    rbp_df = pd.read_excel(xlsx_path, sheet_name="Table S2")
    rbp_genes = set(rbp_df["Gene name"].dropna().astype(str).str.upper().str.strip())
    print(f"Loaded {len(rbp_genes)} RBP genes from file")
    return rbp_genes

In [ ]:
def compute_concentration_data(
    gene_df: pd.DataFrame,
    geneset: Set[str],
    rank_col: str = "z_score",
    ascending: bool = False,
    n_perm: int = 1000,
    k_points: int = 120,
    min_k: int = 50,
    max_k: int = None,
    random_state: int = 0,
) -> Dict:
    rng = np.random.default_rng(random_state)

    df_sorted = (
        gene_df[['gene', rank_col]]
        .dropna(subset=[rank_col])
        .sort_values(rank_col, ascending=ascending)
        .reset_index(drop=True)
    )
    genes = df_sorted['gene'].astype(str).str.upper().to_numpy()

    gs_upper = set(g.upper() for g in geneset)
    is_member = np.isin(genes, list(gs_upper))

    N = len(genes)
    n_hit = int(is_member.sum())
    p0 = n_hit / N

    effective_max = min(max_k, N) if max_k else N
    if max_k:
        k_grid = np.unique(np.linspace(min_k, effective_max, k_points).astype(int))
    else:
        k_grid = np.unique(
            np.rint(np.geomspace(max(min_k, 10), effective_max, k_points)).astype(int)
        )
    k_grid = k_grid[(k_grid >= min_k) & (k_grid <= effective_max)]

    csum = np.cumsum(is_member)
    prec_obs = csum[k_grid - 1] / k_grid

    prec_perm = np.empty((n_perm, len(k_grid)), dtype=float)
    for i in range(n_perm):
        shuf = rng.permutation(is_member)
        csum_p = np.cumsum(shuf)
        prec_perm[i] = csum_p[k_grid - 1] / k_grid

    lo, hi = np.percentile(prec_perm, [2.5, 97.5], axis=0)

    def _trapez(y):
        return np.trapz(y, x=k_grid) / (k_grid[-1] - k_grid[0])

    y_obs = np.clip(prec_obs - p0, 0, None)
    y_perm = np.clip(prec_perm - p0, 0, None)
    aulc_obs = _trapez(y_obs)
    aulc_perm = np.apply_along_axis(_trapez, 1, y_perm)
    pval = (np.sum(aulc_perm >= aulc_obs) + 1) / (len(aulc_perm) + 1)

    return {
        "k_grid": k_grid,
        "prec_obs": prec_obs,
        "lo": lo,
        "hi": hi,
        "p0": p0,
        "aulc_obs": aulc_obs,
        "p_perm": pval,
        "n_genes": N,
        "n_hit": n_hit,
        "max_k": effective_max,
    }

def plot_two_tailed_concentration(
    gene_df: pd.DataFrame,
    geneset: Set[str],
    geneset_name: str = "RBP",
    rank_col: str = "z_score",
    n_perm: int = 1000,
    k_points: int = 120,
    min_k: int = 50,
    max_k: int = None,
    random_state: int = 42,
    figsize: Tuple = (7.5, 5.5),
    title: str = None,
    save_path: str = None,
) -> Tuple[plt.Figure, Dict, Dict]:
    if title is None:
        zoom_str = f" (top {max_k:,})" if max_k else ""
        title = f"SLE vs HC: {geneset_name} concentration at both tails{zoom_str}"

    print(f"Computing SLE-enriched tail (descending)...")
    sle_data = compute_concentration_data(
        gene_df, geneset, rank_col=rank_col, ascending=False,
        n_perm=n_perm, k_points=k_points, min_k=min_k, max_k=max_k,
        random_state=random_state,
    )

    print(f"Computing HC-enriched tail (ascending)...")
    hc_data = compute_concentration_data(
        gene_df, geneset, rank_col=rank_col, ascending=True,
        n_perm=n_perm, k_points=k_points, min_k=min_k, max_k=max_k,
        random_state=random_state + 1,
    )

    p0 = sle_data["p0"]

    fig, ax = plt.subplots(figsize=figsize)

    ax.fill_between(
        sle_data["k_grid"], sle_data["lo"], sle_data["hi"],
        color="#DDDDDD", alpha=0.6, label="Permutation 95% band", zorder=1,
    )

    ax.plot(
        sle_data["k_grid"], sle_data["prec_obs"],
        lw=2.2, color="#EE6352", label="SLE-enriched tail (high z-score)",
        zorder=3,
    )

    ax.plot(
        hc_data["k_grid"], hc_data["prec_obs"],
        lw=2.2, color="#4A90D9", label="HC-enriched tail (low z-score)",
        zorder=3,
    )

    ax.axhline(
        p0, ls="--", lw=1.2, color="#7A7A7A",
        label=f"Baseline (p\u2080={p0:.3f})", zorder=2,
    )

    ax.set_xlabel("Top K genes (from each tail)", fontsize=11)
    ax.set_ylabel(f"Fraction {geneset_name}s among top-K", fontsize=11)
    ax.set_title(title, fontsize=12, pad=8)

    ax.set_xlim(sle_data["k_grid"][0], sle_data["k_grid"][-1])

    y_max = max(
        sle_data["hi"].max(), sle_data["prec_obs"].max(),
        hc_data["hi"].max(), hc_data["prec_obs"].max(),
    )
    ax.set_ylim(0, y_max * 1.08)

    ax.legend(frameon=True, framealpha=0.9, edgecolor="gray", loc="upper right", fontsize=9)

    stats_text = (
        f"SLE tail: perm p = {sle_data['p_perm']:.3g}\n"
        f"HC tail:  perm p = {hc_data['p_perm']:.3g}\n"
        f"N = {sle_data['n_genes']:,}, {geneset_name}s = {sle_data['n_hit']:,}"
    )
    ax.text(
        0.02, 0.02, stats_text,
        transform=ax.transAxes, ha='left', va='bottom',
        fontsize=9, family='monospace',
        bbox=dict(boxstyle='round,pad=0.3', facecolor='white', edgecolor='gray', alpha=0.85),
    )

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(True, linestyle='--', alpha=0.15, zorder=0)

    plt.tight_layout()

    if save_path:
        fig.savefig(save_path, format='pdf', bbox_inches='tight')
        print(f"Saved: {save_path}")

        png_path = save_path.replace('.pdf', '.png')
        fig.savefig(png_path, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"Saved: {png_path}")

    plt.show()

    return fig, sle_data, hc_data

In [ ]:
os.makedirs(FIG_DIR, exist_ok=True)

gene_df = load_gene_results(GENE_RESULTS_CSV)
print(f"Loaded {len(gene_df):,} genes")
print(f"Columns: {list(gene_df.columns)}")

rbp_genes = load_rbp_genes(RBP_CELL_LINE_XLSX)
universe = set(gene_df['gene'].dropna())
rbp_in_universe = set(g.upper() for g in rbp_genes) & universe
print(f"RBP in universe: {len(rbp_in_universe)}")

fig_500, sle_500, hc_500 = plot_two_tailed_concentration(
    gene_df=gene_df,
    geneset=rbp_genes,
    geneset_name="RBP",
    rank_col="z_score",
    n_perm=100000,
    k_points=80,
    min_k=20,
    max_k=500,
    random_state=42,
    save_path=os.path.join(FIG_DIR, "slehc_rbp_two_tailed_top500.pdf"),
)

print(f"\nTop 500 results:")
print(f"  SLE tail: AULC = {sle_500['aulc_obs']:.4f}, p = {sle_500['p_perm']:.4g}")
print(f"  HC tail:  AULC = {hc_500['aulc_obs']:.4f}, p = {hc_500['p_perm']:.4g}")